In [1]:
from dotenv import load_dotenv

load_dotenv()


True

# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. 
In particular: 
1) added error handling for MCP failures, 
2) limited the number of searches. It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

## Setup Tools


In [2]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    # Returns a CallToolResult with an error message if all retries are exhausted.
    # MCP tool calls are expected to return a CallToolResult, so we return a CallToolResult with an error message instead of raising an exception.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [3]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient() #Tavily API key is automatically loaded from the environment variable TAVILY_API_KEY. [Refer to Email from Rotem Weiss - Tavily Team : Quickstart Guide for more information]   

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

/var/folders/25/43cl488n18jff_vg4p7jy9gc0000gn/T/ipykernel_7666/2549610831.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Create State

In [5]:
from langchain.agents import AgentState

# Data type annotations for the state of the wedding planning agent. This is used to store the state of the agent between calls.
# Data type annotations are used to define the structure of the state object, which can be used to store information about the wedding planning process, such as the origin and destination of the wedding, the number of guests, and the genre of the wedding.
class WeddingState(AgentState):
    origin: str 
    destination: str
    guest_count: str
    genre: str

## Create Subagents


# Create Subagents: Travel Agent, Venue Agent, and Playlist Agent. 
## Worker agents that the will execute specific tasks and report back to the Supervisor Agent.

In [6]:
from langchain.agents import create_agent

# ====== Travel Agent ======
# Travel agent is a specialized agent that searches for flights to the desired destination wedding location. 
# It uses the Tavily API to search for flights based on the criteria of price, duration, and date. 
# The agent is designed to find the best flight options without asking any follow-up questions, and it may make multiple searches to iteratively find the best options. 
# Once the best options are found, the agent will provide a shortlist of options to the user.

travel_agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [7]:
# ====== Venue agent ======
# Venue agent is a specialized agent that searches for venues in the desired location and with the desired capacity.
# It uses the Tavily API to search for venues based on the criteria of price, capacity, and reviews. 
# The agent is designed to find the best venue options without asking any follow-up questions, and it may make multiple searches to iteratively find the best options. 
# Once the best options are found, the agent will provide a shortlist of options to the user.

venue_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [8]:
# ====== Playlist agent ======
# Playlist agent is a specialized agent that queries a SQLite database to curate the perfect playlist for a wedding based on a given genre.
# It uses the Tavily API to search for venues based on the criteria of price, capacity, and reviews.
# The agent is designed to find the best playlist options without asking any follow-up questions, and it may make multiple searches to iteratively find the best options.
# Once the best options are found, the agent will provide a shortlist of options to the user.

playlist_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [9]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command
# ====== Flight agent ======
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content
# ====== Venue agent ======
@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content
# ====== Playlist agent ======
@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content
# ===== Update state tool ======
@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [10]:
from langchain.agents import create_agent
# ====== Wedding Coordinator agent ======
# Wedding coordinator agent is a specialized agent that coordinates the wedding planning process by delegating tasks to the travel agent, venue agent, and playlist agent. 
# It uses the Tavily API to search for flights, venues, and playlists based on the criteria of price, duration, date, capacity, and genre. 
# The agent is designed to find the best options without asking any follow-up questions, and it may make multiple searches to iteratively find the best options. 
# Once the best options are found, the agent will provide a shortlist of options to the user and coordinate the perfect wedding.    

coordinator = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test


In [11]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from Dallas and I'd like a wedding in New York for 100 guests, Hip-hop-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [12]:
from pprint import pprint

pprint(response)

{'destination': 'New York',
 'genre': 'Hip-hop',
 'guest_count': '100',
 'messages': [HumanMessage(content="I'm from Dallas and I'd like a wedding in New York for 100 guests, Hip-hop-genre", additional_kwargs={}, response_metadata={}, id='90830cea-9593-45bd-b438-4a95eac61ac7'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 422, 'prompt_tokens': 338, 'total_tokens': 760, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EW2KRW0q2JGfcl3hoAEAG3j37t4R8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1120c-3fd4-7633-a223-540523ca8351-0', tool_calls=[{'name': 'update_state'

In [13]:
print(response["messages"][-1].content)

Fantastic start. I’ve organized the initial options for travel, venues, and a hip-hop playlist to match your 100-guest New York wedding. Here’s a concise plan you can review, with quick pros/cons and next-step questions.

Origin: Dallas | Destination: New York | Guests: 100 | Genre vibe: Hip-hop

1) Travel options (Dallas to New York area, wedding window around mid-May 2027)
- Best overall value and timing (nonstop, ~3h15): DFW → EWR, May 11 or May 12, around 10:00–14:15
  - May 11: 10:00–14:15, 255 USD, nonstop
    Booking: https://kiwi.com/u/shwxgs
  - May 12: 10:00–14:15, 255 USD, nonstop
    Booking: https://kiwi.com/u/zsnkkr
- Alternative with similar timings: May 11, 13:50–18:09, 255 USD
  - Booking: https://kiwi.com/u/ddgsxh
- Additional note: Newark (EWR) is the most straightforward NYC option for quick NYC access. If you’d prefer JFK or LGA, I can search those too.

2) Venue options in New York for 100 guests
A quick take on solid 100-person-capacity venues with price ranges a

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r